In [14]:
import json,csv
import requests
import os.path
import pandas as pd
import numpy as np

bic_etl_home = os.getenv('bic_etl_home')

In [15]:
def getActivityLog():
    flog=open(f"{bic_etl_home}/general/datasync/config.json")
    info = json.load(flog)
    username=info['username']
    password=info['password']
    api=info['appToken']
#    "$where": "created_at > '2023-01-01T00:00:00' and acting_user_name = 'Colorado Information Marketplace' order by 'created_at' desc"
    
    # URL of the login form
    query = {
    "$where": "created_at > '2023-01-01T00:00:00' and (acting_user_name = 'Colorado Information Marketplace' or acting_user_name = 'Business Intelligence Center of CO')"
    }
    login_url = 'https://data.colorado.gov/api/activity_log.json?$limit=6000000'
    response=requests.get(login_url,auth=(username, password),params=query)
    if response.status_code == 200:
      
#        activity_log=json.loads(response.text)
        print(type(response.text))
    else:
        print("Failed to download Activity Log")
        print(f"Error: {response.status_code}")
        print(f"Message: {response.text}")
    return pd.DataFrame(json.loads(response.text))


dfActivity = getActivityLog()

<class 'str'>


In [ ]:
import tkinter as tk
from tkinter import ttk
import pandas as pd
import tkinter.font as tkFont

# Example DataFrame
needs= ['affected_item', 'created_at', 'activity_type',
       'acting_user_name', 'service', 'dataset_uid', 
       'dataset_name','asset_type', 'details']


def show_dataframe(title):
    print(title)
    dfOut=dfActivity.loc[dfActivity['affected_item'] == title,needs]

    # Clear previous table
    for row in tree.get_children():
        tree.delete(row)

    # Insert new data
    for _, row in dfOut.iterrows():
        print(row)
    #    tree.insert("", tk.END, values=list(row))
        tree.insert("", tk.END, values=list(row), tags=("body",))


root = tk.Tk()
root.title("DataFrame Viewer (Treeview Table)")
root.geometry("900x300")

# Dropdown (sniffer-style)
titles=dfActivity['dataset_name'].unique()

titles=[title for title in titles if isinstance(title,str)]
titles.sort()
dropdown_var = tk.StringVar(value=titles[0])
ttk.Label(root, text="Choose an action:").pack(pady=5)
#dropdown = ttk.OptionMenu(root, dropdown_var, *options)
#dropdown = ttk.OptionMenu(root, dropdown_var, options[0], *options)
dropdown = ttk.OptionMenu(root, dropdown_var, titles[0], *titles, command=lambda _: on_option_change())

dropdown.pack()

# Treeview setup
cols = list(dfActivity.columns)
tree = ttk.Treeview(root, columns=cols, show="headings")
# Define custom fonts
tree_font = tkFont.Font(family="Consolas", size=8)
# header_font = tkFont.Font(family="Consolas", size=14, weight="bold")
# fonts + style (do this before creating the Treeview)
fixed = tkFont.nametofont("TkFixedFont")
fixed.configure(size=6)  # ← change size here
header = tkFont.Font(family=fixed.cget("family"), size=14, weight="bold")

style = ttk.Style(root)
style.theme_use('clam')  # ensures style changes take effect across platforms
style.configure("Treeview", font=fixed, rowheight=26)           # row font + height
style.configure("Treeview.Heading", font=header)
# Apply font styles
tree.tag_configure("body", font=tree_font)

# style = ttk.Style()
# style.configure("Treeview.Heading", font=header_font)
# style = ttk.Style()
# style.configure("Treeview", font=("Consolas", 8))         # body text
# style.configure("Treeview.Heading", font=("Consolas", 8, "bold"))  # header text
for col in cols:
    tree.heading(col, text=col)
    tree.column(col, width=100, anchor="center")

tree.pack(expand=True, fill="both")

# Scrollbar
scrollbar = ttk.Scrollbar(root, orient="vertical", command=tree.yview)
scrollbar.pack(side="right", fill="y")
tree.configure(yscrollcommand=scrollbar.set)

# Event binding
def on_option_change(*args):
        title = dropdown_var.get()
        show_dataframe(title)

dropdown_var.trace_add("write", on_option_change)

root.mainloop()


In [22]:
import tkinter as tk
from tkinter import ttk
import pandas as pd
import webbrowser
import tempfile

# --- Sample Data ---
import os, platform, webbrowser, subprocess

def open_html(path):
    system = platform.system().lower()
    url = "file://" + path
    print("SYSTEM",system)
    try:
        if system == "windows":
            webbrowser.get("windows-default").open(url)
        elif system == "darwin":  # macOS
            subprocess.run(["open", path])
        elif system == "linux":
            # Try xdg-open first
            if subprocess.call(["which", "xdg-open"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0:
                subprocess.Popen(["xdg-open", path])
            else:
                print(f"Open manually in your browser:\n{url}")
        else:
            print(f"Unknown OS. Open manually:\n{path}")
    except webbrowser.Error:
        print(f"Could not locate a runnable browser.\nOpen manually:\n{path}")

needs = ['affected_item', 'created_at', 'activity_type',
         'acting_user_name', 'service', 'dataset_uid',
         'dataset_name', 'asset_type', 'details']

titles=dfActivity['dataset_name'].unique()
titles=[title for title in titles if isinstance(title,str)]
titles.sort()



# --- Functions ---
def open_dataframe_in_browser(df):
    html = df.to_html(index=False, justify="center", escape=False)

    # Create a temp HTML file
    with tempfile.NamedTemporaryFile("w", delete=False, suffix=".html") as f:
        f.write("""
        <html>
        <head>
        <style>
            body { font-family: Arial; padding: 20px; }
            table { border-collapse: collapse; width: 100%; table-layout: fixed; }
            th, td {
                border: 1px solid #ccc;
                padding: 8px;
                text-align: left;
                word-wrap: break-word;
                white-space: normal;
            }
            th { background-color: #f0f0f0; }
        </style>
        </head>
        <body>
        <h2>DataFrame Viewer</h2>
        """ + html + "</body></html>")
        path = f.name

    #webbrowser.get("chrome").open("file://" + path)
    open_html(path)

def on_option_change(*args):
    title = dropdown_var.get()
    dfOut = dfActivity.loc[dfActivity['affected_item'] == title,needs]
    open_dataframe_in_browser(dfOut)

# --- GUI ---
root = tk.Tk()
root.title("DataFrame Browser Viewer")
root.geometry("400x150")


dropdown_var = tk.StringVar(value=titles[0])

ttk.Label(root, text="Choose an action:").pack(pady=10)
dropdown = ttk.OptionMenu(root, dropdown_var, titles[0], *titles,
                          command=lambda _: on_option_change())
dropdown.pack(pady=10)

ttk.Label(root, text="Selecting 'Show Table' will open your browser.").pack(pady=5)

root.mainloop()


SYSTEM linux
Open manually in your browser:
file:///tmp/tmph6kc80h1.html


In [ ]:
import streamlit as st
import pandas as pd

# ---- Sample DataFrame (replace this with your real dfActivity) ----


# ---- Filter ----
needs = ['affected_item', 'created_at', 'activity_type',
         'acting_user_name', 'service', 'dataset_uid',
         'dataset_name', 'asset_type', 'details']


# ---- Streamlit UI ----
st.set_page_config(page_title="DataFrame Viewer", layout="wide")

st.title("📊 DataFrame Viewer")

# Dropdown menu (replaces Tkinter OptionMenu)
options = ["Show Table", "Summary", "Other"]
title = st.selectbox("Choose an action:", titles)

# Event-like logic (replaces trace_add / event sniffer)
dfOut = dfActivity.loc[dfActivity['affected_item'] == title, needs]

st.write("### Full Table")
st.dataframe(dfOut, use_container_width=True)

st.info("Select an option to view data.")

